In [ ]:
import os
import sys

sys.path.append("kaggle/input/polymer_pipeline")

In [2]:
from data_preparation import get_data_paths, load_and_split_data
import model

In [ ]:
os.environ["NEURIPS_DATA_PATH"] = "kaggle/input/neurips-open-polymer-prediction-2025"
os.environ["EXTRA_DATA_BASE"] = "kaggle/input/smiles-extra-data"
os.environ["TC_DATA_BASE"] = "kaggle/input/tc-smiles"

In [4]:
paths = get_data_paths()
for k, v in paths.items():
    print(f"{k}: {v}")

train_csv: kaggle/input/neurips-open-polymer-prediction-2025/train.csv
test_csv: kaggle/input/neurips-open-polymer-prediction-2025/test.csv
sample_submission: kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
tc_data: kaggle/input/tc-smiles/Tc_SMILES.csv
tg_jcim_data: kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv
tg_excel_data: kaggle/input/smiles-extra-data/data_tg3.xlsx
density_data: kaggle/input/smiles-extra-data/data_dnst1.xlsx
supplement_dir: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement
ffv_data: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
dataset1: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
dataset2: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
dataset3: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [5]:
train_df, val_df, test_df = load_and_split_data(paths)
print("Loaded:", len(train_df), len(val_df), len(test_df))

原始训练: 7973 条
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737 | 填充: 0
新增样本: 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526 | 填充: 15
新增样本: 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0 | 填充: 0
新增样本: 499 条
  → 正在增强 Density 数据，共 787 条


[10:50:11] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[10:50:11] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[10:50:11] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[10:50:11] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[10:50:11] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[10:50:11] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[10:50:11] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[10:50:11] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[10:50:11] SMILES Parse 

cross_smiles: 254 | 填充: 110
新增样本: 525 条
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43 | 填充: 43
新增样本: 819 条
Loaded: 8064 1008 1009


In [6]:
import torch
import pandas as pd
from train_stage2 import optimize_stage2, train_final_stage2_model

/usr/local/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
train_df.head()

,id,SMILES,Tg,FFV,Tc,Density,Rg
0,2.026603e+09,*Oc1ccc(CC2(Cc3ccc(*)cc3)c3ccccc3-c3ccccc32)cc1,NaN,0.386736,NaN,NaN,NaN
1,1.189403e+09,*CC(O)COc1ccc(C(C)CC(C)(C)c2ccc(O*)cc2)cc1,NaN,0.354235,NaN,NaN,NaN
2,1.686537e+09,*C(=O)c1ccc2c(c1)C(=O)N(c1c(C)cc(C(c3cc(C)c(N4...,NaN,0.430396,NaN,NaN,NaN
3,2.632934e+08,*Oc1ccc2ccc(Oc3ccc(C(=Nc4ccc(N=C(c5ccccc5)c5cc...,NaN,0.381740,NaN,NaN,NaN
4,1.280165e+09,*Nc1ccc(-c2ccc(N*)c(OC)c2)cc1OC,NaN,0.334115,NaN,NaN,NaN


In [ ]:
study = optimize_stage2(
    train_df=train_df,
    stage1_model_path="stage1_final_model.pth",
    study_name="stage2_graph_ssl",
    storage_uri="sqlite:///stage2_optuna.db",
    n_trials=50,
    tmp_dir="tmp_stage2",
    output_dir="stage2_artifacts",
)

📦 构建 PolymerDataset，样本数=8064


   成功转换为图数据: 8064 条
using device: cuda


[I 2025-08-02 10:50:34,759] A new study created in RDB with name: stage2_graph_ssl


Epoch 1: avg_loss=4.9299e+01, best_loss=4.9299e+01 (no_improve=0)
Epoch 2: avg_loss=5.0659e+00, best_loss=5.0659e+00 (no_improve=0)
Epoch 3: avg_loss=7.0294e+00, best_loss=5.0659e+00 (no_improve=1)
Epoch 4: avg_loss=3.1385e+00, best_loss=3.1385e+00 (no_improve=0)
Epoch 5: avg_loss=4.5251e+00, best_loss=3.1385e+00 (no_improve=1)
Epoch 6: avg_loss=2.8178e+00, best_loss=2.8178e+00 (no_improve=0)
Epoch 7: avg_loss=4.1417e+00, best_loss=2.8178e+00 (no_improve=1)
Epoch 8: avg_loss=5.4807e+00, best_loss=2.8178e+00 (no_improve=2)
Epoch 9: avg_loss=6.1450e+00, best_loss=2.8178e+00 (no_improve=3)
Epoch 10: avg_loss=4.0636e+00, best_loss=2.8178e+00 (no_improve=4)
Epoch 11: avg_loss=2.1844e+00, best_loss=2.1844e+00 (no_improve=0)
Epoch 12: avg_loss=5.5996e+00, best_loss=2.1844e+00 (no_improve=1)
Epoch 13: avg_loss=4.1557e+00, best_loss=2.1844e+00 (no_improve=2)
Epoch 14: avg_loss=4.6570e+00, best_loss=2.1844e+00 (no_improve=3)
Epoch 15: avg_loss=4.5974e+00, best_loss=2.1844e+00 (no_improve=4)
Epoc

[I 2025-08-02 10:51:17,075] Trial 0 finished with value: 2.1844423460581948 and parameters: {'predictor_hidden_dim': 128, 'predictor_num_layers': 2, 'lr': 0.0009820954911518615}. Best is trial 0 with value: 2.1844423460581948.


...

Early stopping at epoch 46
Epoch 1: avg_loss=7.0514e+01, best_loss=7.0514e+01 (no_improve=0)
Epoch 2: avg_loss=2.7655e+00, best_loss=2.7655e+00 (no_improve=0)
Epoch 3: avg_loss=2.5754e+00, best_loss=2.5754e+00 (no_improve=0)
Epoch 4: avg_loss=1.7701e+00, best_loss=1.7701e+00 (no_improve=0)
Epoch 5: avg_loss=1.7657e+00, best_loss=1.7657e+00 (no_improve=0)
Epoch 6: avg_loss=1.8744e+00, best_loss=1.7657e+00 (no_improve=1)
Epoch 7: avg_loss=1.6155e+00, best_loss=1.6155e+00 (no_improve=0)
Epoch 8: avg_loss=2.5750e+00, best_loss=1.6155e+00 (no_improve=1)
Epoch 9: avg_loss=2.0377e+00, best_loss=1.6155e+00 (no_improve=2)


[I 2025-08-02 11:14:06,830] Trial 44 pruned. 
[I 2025-08-02 11:14:08,434] Trial 45 pruned. 


Epoch 1: avg_loss=8.8870e+01, best_loss=8.8870e+01 (no_improve=0)


[I 2025-08-02 11:14:11,495] Trial 46 pruned. 
[I 2025-08-02 11:14:12,966] Trial 47 pruned. 
[I 2025-08-02 11:14:14,208] Trial 48 pruned. 
[I 2025-08-02 11:14:15,789] Trial 49 pruned. 


✅ Best Stage2 model (trial 3) saved to stage2_artifacts


In [10]:
# 加载最佳参数
params = torch.load("stage2_artifacts/stage2_best_params_trial3.pt")  # 根据实际 trial 号改名
params

{'lr': 3.741017631659677e-05,
 'hidden_dim': 256,
 'num_edge_layers': 2,
 'trial_number': 6,
 'predictor_hidden_dim': 128,
 'predictor_num_layers': 1}

In [12]:
train_final_stage2_model(
    train_df=train_df,
    stage1_model_path="stage1_final_model.pth",
    best_params_path="stage2_artifacts/stage2_best_params_trial3.pt",
    n_epochs=1000,
    patience=20
)


✅ Loaded best stage2 params from stage2_artifacts/stage2_best_params_trial3.pt
📦 构建 PolymerDataset，样本数=8064
   成功转换为图数据: 8064 条
Epoch 1: avg_loss=3.9524e+02, best_loss=3.9524e+02 (no_improve=0)
Epoch 2: avg_loss=2.0204e+02, best_loss=2.0204e+02 (no_improve=0)
Epoch 3: avg_loss=4.9380e+01, best_loss=4.9380e+01 (no_improve=0)
Epoch 4: avg_loss=1.0625e+01, best_loss=1.0625e+01 (no_improve=0)
Epoch 5: avg_loss=8.3529e+00, best_loss=8.3529e+00 (no_improve=0)
Epoch 6: avg_loss=6.8673e+00, best_loss=6.8673e+00 (no_improve=0)
Epoch 7: avg_loss=5.9052e+00, best_loss=5.9052e+00 (no_improve=0)
Epoch 8: avg_loss=5.2469e+00, best_loss=5.2469e+00 (no_improve=0)
Epoch 9: avg_loss=4.6738e+00, best_loss=4.6738e+00 (no_improve=0)
Epoch 10: avg_loss=4.3486e+00, best_loss=4.3486e+00 (no_improve=0)
Epoch 11: avg_loss=3.7567e+00, best_loss=3.7567e+00 (no_improve=0)
Epoch 12: avg_loss=3.4119e+00, best_loss=3.4119e+00 (no_improve=0)
Epoch 13: avg_loss=3.1363e+00, best_loss=3.1363e+00 (no_improve=0)
Epoch 14: 

GraphSSLModel(
  (encoder): WDMPNN()
  (predictor): GraphPredictor(
    (mlp): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=1, bias=True)
    )
  )
)